In [5]:
import hashlib

import json
def sha256(data):
    """Returns the SHA-256 hash of the given data."""
    return hashlib.sha256(data.encode('utf-8')).hexdigest()

def build_merkle_tree(transactions):
    """Builds the Merkle tree and returns the Merkle root."""
    leaves = [sha256(tx) for tx in transactions]
    while len(leaves) > 1:
        new_leaves = []
        for i in range(0, len(leaves), 2):
            left = leaves[i]
            right = leaves[i + 1] if i + 1 < len(leaves) else leaves[i]
            new_leaves.append(sha256(left + right))
        leaves = new_leaves
    return leaves[0]

In [6]:
transactions = ["tx1", "tx2", "tx3", "tx4"]
root = build_merkle_tree(transactions)
print(f"Merkle Root: {root}")

Merkle Root: 773bc304a3b0a626a520a8d6eacc36809ac18c0b174f3ff3cdaf0a4e9c64433d


In [7]:
def merkle_proof(transactions: list[str], target_tx: str) -> list[str]:
    """Generate a Merkle proof using sorted-pair hashing."""
    target_hash = sha256(target_tx)
    leaves = [sha256(tx) for tx in transactions]
    
    if target_hash not in leaves:
        raise ValueError("Target transaction not found in transactions.")
        
    proof = []
    index = leaves.index(target_hash)
    
    while len(leaves) > 1:
        if index % 2 == 0:
            sibling = leaves[index + 1] if index + 1 < len(leaves) else leaves[index]
        else:
            sibling = leaves[index - 1]
        proof.append(sibling)
        
        new_leaves = []
        for i in range(0, len(leaves), 2):
            left = leaves[i]
            right = leaves[i + 1] if i + 1 < len(leaves) else leaves[i]
            
            if left < right:
                new_leaves.append(sha256(left + right))
            else:
                new_leaves.append(sha256(right + left))
                
        leaves = new_leaves
        index //= 2
        
    return proof

def verify_merkle_proof(proof: list[str], target_tx: str, root: str) -> bool:
    """Verify a Merkle proof using sorted-pair hashing."""
    computed_hash = sha256(target_tx)
    
    for sibling in proof:
        if computed_hash < sibling:
            computed_hash = sha256(computed_hash + sibling)
        else:
            computed_hash = sha256(sibling + computed_hash)
            
    return computed_hash == root

In [8]:
transactions = ["tx1", "tx2", "tx3", "tx4"]
root = build_merkle_tree(transactions)
proof = merkle_proof(transactions, "tx3")
print(f"Merkle Proof for 'tx3': {proof}")
print(f"Verification: {verify_merkle_proof(proof, 'tx3', root)}")

Merkle Proof for 'tx3': ['41b637cfd9eb3e2f60f734f9ca44e5c1559c6f481d49d6ed6891f3e9a086ac78', '472b125cd7be33e50ea6ef56dd83389f4ab7a5e1907c082d0d5bfa86431c953a']
Verification: False


In [9]:
import hashlib
import datetime
def sha256(data):
    """Returns the SHA-256 hash of the given data."""
    return hashlib.sha256(data.encode('utf-8')).hexdigest()
class Block:
    def __init__(self, index, timestamp, transactions, previous_hash):
        self.index = index
        self.timestamp = timestamp
        self.transactions = transactions
        self.previous_hash = previous_hash
        self.merkle_root = build_merkle_tree(transactions) # Store Merkle root
        self.hash = self.calculate_hash()
    def calculate_hash(self):
        """Generate the block's hash."""
        block_string = str(self.index) + str(self.timestamp) + self.merkle_root + self.previous_hash
        return sha256(block_string)
class Blockchain:
    def __init__(self):
        self.chain = [self.create_genesis_block()]
    def create_genesis_block(self):
        return Block(0, str(datetime.datetime.now()), ["Genesis Block"], "0")
    def add_block(self, transactions):
        last_block = self.chain[-1]
        new_block = Block(len(self.chain), str(datetime.datetime.now()),
        transactions, last_block.hash)
        self.chain.append(new_block)
        # Example usage
        blockchain = Blockchain()
        blockchain.add_block(["tx1", "tx2", "tx3"])
        blockchain.add_block(["tx4", "tx5"])
        for block in blockchain.chain:
            print(f"Block #{block.index}: {block.hash}")
            print(f"Merkle Root: {block.merkle_root}")
            print(f"Previous Hash: {block.previous_hash}")
            print()